# Gold Daily Electricity Consumption by Price Area

Create daily consumption totals for each Norwegian electricity price area and consumer group using the Silver Elhub price-area dataset.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from gold_layer.consumption import ingest

spark = (SparkSession.builder
    .appName("Gold Daily Consumption")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

26/11/22 15:53:03 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/22 15:53:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 15:53:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Build Gold consumption

In [2]:
result = ingest(
    silver_root=PROJECT_ROOT / "data/silver",
    gold_root=PROJECT_ROOT / "data/gold",
    start="2024-01-01",
    end="2026-01-01",
    spark=spark,
)

print(result)

daily = spark.read.parquet(str(PROJECT_ROOT / "data/gold/consumption/daily"))
daily.printSchema()
daily.orderBy("date", "price_area", "consumption_group").show(10, truncate=False)

{'dataset': 'daily_consumption', 'status': 'success', 'rows': 18275, 'gold_path': '/home/roland/UiS/rewrite/group14/data/gold/consumption/daily'}
root
 |-- date: date (nullable = true)
 |-- price_area: string (nullable = true)
 |-- consumption_group: string (nullable = true)
 |-- consumption_kwh: decimal(28,6) (nullable = true)
 |-- hour_count: integer (nullable = true)
 |-- expected_hours: integer (nullable = true)
 |-- is_complete: boolean (nullable = true)

+----------+----------+-----------------+---------------+----------+--------------+-----------+
|date      |price_area|consumption_group|consumption_kwh|hour_count|expected_hours|is_complete|
+----------+----------+-----------------+---------------+----------+--------------+-----------+
|2024-01-01|NO1       |cabin            |4052234.671000 |24        |24            |true       |
|2024-01-01|NO1       |household        |69182787.521000|24        |24            |true       |
|2024-01-01|NO1       |primary          |1735314.837000

## Check row, keys, nulls and coverage

In [3]:
rows = daily.count()

unique_keys = daily.select("date", "price_area", "consumption_group").distinct().count()
required = ["date", "price_area", "consumption_group", "consumption_kwh", "hour_count"]
null_rows = daily.filter(
    F.expr(" OR ".join(f"{col} IS NULL" for col in required))
).count()
incomplete = daily.filter(~F.col("is_complete")).count()

print(f"Rows: {rows:,} (expected 18,275)")
print(f"Duplicate keys: {rows - unique_keys}")
print(f"Rows with null required values: {null_rows}")
print(f"Incomplete daily records: {incomplete}")

Rows: 18,275 (expected 18,275)
Duplicate keys: 0
Rows with null required values: 0
Incomplete daily records: 0


## Coverage by price area and consumer group

In [4]:
daily.groupBy("price_area", "consumption_group").agg(
    F.count("*").alias("days"),
    F.sum("hour_count").alias("hours"),
    F.sum("consumption_kwh").alias("total_kwh"),
).orderBy("price_area", "consumption_group").show(30, truncate=False)

+----------+-----------------+----+-----+------------------+
|price_area|consumption_group|days|hours|total_kwh         |
+----------+-----------------+----+-----+------------------+
|NO1       |cabin            |731 |17544|1498695792.867000 |
|NO1       |household        |731 |17544|30785091759.626000|
|NO1       |primary          |731 |17544|785652225.889000  |
|NO1       |secondary        |731 |17544|12481492170.323000|
|NO1       |tertiary         |731 |17544|19761521691.256000|
|NO2       |cabin            |731 |17544|1202633826.404000 |
|NO2       |household        |731 |17544|17603872041.568000|
|NO2       |primary          |731 |17544|1363273539.222000 |
|NO2       |secondary        |731 |17544|39740107041.983000|
|NO2       |tertiary         |731 |17544|10351359905.395000|
|NO3       |cabin            |731 |17544|789950169.296000  |
|NO3       |household        |731 |17544|11991463565.842000|
|NO3       |primary          |731 |17544|1621566684.915000 |
|NO3       |secondary   